# 02 — Bronze Ingestion: Customers

In [0]:
from pyspark.sql import functions as F
import uuid

S3_BUCKET = "s3a://insurance-lakehouse-project"
RAW_BASE_PATH = f"{S3_BUCKET}/raw"
CHECKPOINT_BASE_PATH = f"{S3_BUCKET}/checkpoints"
CATALOG_NAME = "insurance_lakehouse"
BRONZE_SCHEMA = "bronze"

FULL_RELOAD = True  # set False for incremental production runs

schema_hints = {
    "customers":        "gdpr_consent BOOLEAN",
    "claims":           "fraud_flag BOOLEAN",
    "agents":           "active_flag BOOLEAN",
    "fraud_indicators": "suspicious_amount_flag BOOLEAN, duplicate_claim_flag BOOLEAN, late_report_flag BOOLEAN, high_risk_region_flag BOOLEAN",
}

datasets = ["customers", "policies", "claims", "payments", "agents", "fraud_indicators"]

if FULL_RELOAD:
    dbutils.fs.rm(f"{CHECKPOINT_BASE_PATH}/", recurse=True)
    print("Checkpoints cleared")
    for name in datasets:
        bronze_table = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.bronze_{name}"
        spark.sql(f"TRUNCATE TABLE {bronze_table}")
        print(f"Truncated {bronze_table}")

ingest_run_id = str(uuid.uuid4())
print(f"\nRun id: {ingest_run_id}")

results = []

for name in datasets:
    raw_path = f"{RAW_BASE_PATH}/{name}"
    checkpoint_path = f"{CHECKPOINT_BASE_PATH}/{name}"
    bronze_table = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.bronze_{name}"

    print(f"\nIngesting {name}...")
    print(f"Raw path:        {raw_path}")
    print(f"Checkpoint path: {checkpoint_path}")
    print(f"Bronze table:    {bronze_table}")

    # capture count before for incremental validation
    bronze_count_before = spark.table(bronze_table).count()

    reader = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", checkpoint_path)
        .option("header", True)
        .option("inferSchema", True)
    )

    if name in schema_hints:
        reader = reader.option("cloudFiles.schemaHints", schema_hints[name])

    (
        reader
        .load(raw_path)
        .withColumn("ingest_timestamp", F.current_timestamp())
        .withColumn("ingest_run_id", F.lit(ingest_run_id))
        .withColumn("source_file_name", F.col("_metadata.file_path"))
        .writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(bronze_table)
        .awaitTermination()
    )

    bronze_count_after = spark.table(bronze_table).count()
    rows_added = bronze_count_after - bronze_count_before

    if FULL_RELOAD:
        raw_count = spark.read.option("header", True).csv(raw_path).count()
        status = "PASS" if raw_count == bronze_count_after else "FAIL"
        results.append((name, raw_path, bronze_table, raw_count, bronze_count_after, rows_added, status))
    else:
        status = "PASS" if rows_added > 0 else "NO_NEW_FILES"
        results.append((name, raw_path, bronze_table, None, bronze_count_after, rows_added, status))

    print(f"{name} - bronze total: {bronze_count_after}, rows added: {rows_added}, status: {status}")

summary = spark.createDataFrame(
    results,
    ["dataset", "raw_path", "bronze_table", "raw_count", "bronze_total", "rows_added", "status"]
)
display(summary)